# Chapter 26: Loop Closure

<a href="../lite/lab/index.html?path=ch26_loop_closure.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

A robot has been driving for 10 minutes. Its odometry has drifted 2 meters.
Then it sees the starting area again and recognizes it. That single
recognition event does not just fix the current position. It propagates
backward, correcting every pose in the trajectory. The entire path snaps
into place like a zipper closing.

This chapter explores the full loop closure pipeline: **detection** (how
to recognize a revisited place), **constraint addition** (how to add the
edge to the graph), and **global correction** (how optimization distributes
the fix). We also show what happens when loop closure goes wrong.

## 26.1 Detection: Recognizing a Previously Visited Place

Loop closure detection asks: "Have I been here before?" There are several
approaches:

- **Feature matching:** Compare current visual features against a database
  of past observations (e.g., bag of visual words).
- **Scan matching:** Compare the current LiDAR scan against stored scans.
- **Descriptor similarity:** Compute a compact descriptor for each place
  and find nearest neighbors.

We will simulate this with a simple **distance threshold** model: if the
robot's true position is close to a previous position, it can detect the
loop closure (with some probability of failure or false positive).

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_descriptors = 50    # simulated place descriptors
desc_dim = 8          # descriptor dimension
noise_level = 0.3     # descriptor noise
# ──────────────────────────────────────────────────────────────────────────────

# Simulate place descriptors along a trajectory
# Places are arranged in a circle; the last place should match the first
angles = np.linspace(0, 2*np.pi, n_descriptors, endpoint=False)
base_descriptors = np.column_stack([np.sin(angles * k) for k in range(1, desc_dim+1)])

# Add noise
observed_desc = base_descriptors + np.random.randn(n_descriptors, desc_dim) * noise_level

# Compute similarity matrix (cosine similarity)
norms = np.linalg.norm(observed_desc, axis=1, keepdims=True)
normalized = observed_desc / np.maximum(norms, 1e-10)
similarity = normalized @ normalized.T

# Mask out nearby poses (temporal neighbors are not loop closures)
temporal_mask = np.ones_like(similarity)
for i in range(n_descriptors):
    for j in range(max(0, i-5), min(n_descriptors, i+6)):
        temporal_mask[i, j] = 0

masked_sim = similarity * temporal_mask

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
im = ax.imshow(similarity, cmap='RdBu_r', vmin=-0.5, vmax=1, interpolation='nearest')
ax.set_xlabel('Place index'); ax.set_ylabel('Place index')
ax.set_title('Place similarity matrix', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

ax = axes[1]
im = ax.imshow(masked_sim, cmap='RdBu_r', vmin=-0.5, vmax=1, interpolation='nearest')
ax.set_xlabel('Place index'); ax.set_ylabel('Place index')
ax.set_title('Masked (temporal neighbors removed)', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8)

# Find top candidates
top_k = 5
flat_idx = np.argsort(masked_sim.ravel())[::-1][:top_k]
candidates = [(idx // n_descriptors, idx % n_descriptors) for idx in flat_idx]

for i, j in candidates:
    ax.plot(j, i, 'o', color='orange', ms=10, mew=2, fillstyle='none')

plt.tight_layout()
plt.show()

print('Top loop closure candidates (pose_i, pose_j, similarity):')
for i, j in candidates:
    print(f'  ({i:3d}, {j:3d}) similarity = {masked_sim[i, j]:.3f}')

**Observation:** The similarity matrix shows high values along the diagonal
(nearby poses are similar) and in the corners (the trajectory loops back).
After masking temporal neighbors, the remaining high-similarity pairs are
loop closure candidates. In practice, these candidates must be **verified**
before adding them to the graph, because false positives are catastrophic.

## 26.2 Constraint Addition: Adding the Loop Closure Edge

Once a loop closure is detected between pose $i$ and pose $j$ (where
$|i - j|$ is large), we need to compute the **relative transform**
$\mathbf{z}_{ij}$ and its **information matrix** $\Omega_{ij}$.

This transform typically comes from scan matching or feature-based
alignment between the two observations. The information matrix reflects
the precision of this alignment.

The loop closure edge is added to the pose graph just like any other
constraint. The difference: it connects two poses that are far apart
in the temporal sequence.

In [ ]:
# Pose graph utilities (from Chapter 25)
def pose2_compose(a, b):
    c = np.cos(a[2]); s = np.sin(a[2])
    return np.array([a[0] + c*b[0] - s*b[1], a[1] + s*b[0] + c*b[1], a[2] + b[2]])

def pose2_inverse(a):
    c = np.cos(a[2]); s = np.sin(a[2])
    return np.array([-c*a[0] - s*a[1], s*a[0] - c*a[1], -a[2]])

def pose2_between(a, b):
    return pose2_compose(pose2_inverse(a), b)

def wrap_angle(a):
    return (a + np.pi) % (2*np.pi) - np.pi

def compute_edge_residual(xi, xj, z_ij):
    z_pred = pose2_between(xi, xj)
    e = z_ij - z_pred
    e[2] = wrap_angle(e[2])
    return e

def compute_edge_jacobians(xi, xj):
    eps = 1e-6
    z0 = pose2_between(xi, xj)
    A = np.zeros((3, 3)); B = np.zeros((3, 3))
    for k in range(3):
        xi_p = xi.copy(); xi_p[k] += eps
        d = pose2_between(xi_p, xj) - z0; d[2] = wrap_angle(d[2])
        A[:, k] = -d / eps
        xj_p = xj.copy(); xj_p[k] += eps
        d = pose2_between(xi, xj_p) - z0; d[2] = wrap_angle(d[2])
        B[:, k] = -d / eps
    return A, B

def pose_graph_optimize(poses_init, edges, n_iter=5):
    n = len(poses_init)
    poses = [p.copy() for p in poses_init]
    costs = []
    for it in range(n_iter):
        dim = 3 * n; H = np.zeros((dim, dim)); b = np.zeros(dim)
        cost = 0.0
        for (i, j, z_ij, Om) in edges:
            e = compute_edge_residual(poses[i], poses[j], z_ij)
            A, B = compute_edge_jacobians(poses[i], poses[j])
            cost += e @ Om @ e
            ri, rj = 3*i, 3*j
            H[ri:ri+3, ri:ri+3] += A.T @ Om @ A
            H[ri:ri+3, rj:rj+3] += A.T @ Om @ B
            H[rj:rj+3, ri:ri+3] += B.T @ Om @ A
            H[rj:rj+3, rj:rj+3] += B.T @ Om @ B
            b[ri:ri+3] += A.T @ Om @ e
            b[rj:rj+3] += B.T @ Om @ e
        costs.append(cost)
        H[:3,:] = 0; H[:,:3] = 0; H[:3,:3] = np.eye(3)*1e6; b[:3] = 0
        dx = np.linalg.solve(H, -b)
        for k in range(n):
            poses[k] += dx[3*k:3*k+3]; poses[k][2] = wrap_angle(poses[k][2])
    cost = sum(compute_edge_residual(poses[i], poses[j], z) @ Om @
               compute_edge_residual(poses[i], poses[j], z)
               for i, j, z, Om in edges)
    costs.append(cost)
    return poses, costs

print('Pose graph optimizer loaded.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(7)
n_poses_lc = 30
sigma_odom_lc = 0.12
sigma_odom_th_lc = 0.05
sigma_detect = 0.03     # loop closure measurement precision
sigma_detect_th = 0.01
# ──────────────────────────────────────────────────────────────────────────────

# Ground truth: circular trajectory
radius_lc = 5.0
gt_angles = np.linspace(0, 2*np.pi, n_poses_lc, endpoint=False)
gt_poses_lc = []
for a in gt_angles:
    gt_poses_lc.append(np.array([radius_lc*np.cos(a), radius_lc*np.sin(a), a + np.pi/2]))

# Build odometry edges
edges_lc = []
odom_lc = [gt_poses_lc[0].copy()]
for i in range(n_poses_lc - 1):
    z_t = pose2_between(gt_poses_lc[i], gt_poses_lc[i+1])
    noise = np.array([np.random.randn()*sigma_odom_lc, np.random.randn()*sigma_odom_lc,
                       np.random.randn()*sigma_odom_th_lc])
    z_n = z_t + noise; z_n[2] = wrap_angle(z_n[2])
    Om_o = np.diag([1/sigma_odom_lc**2, 1/sigma_odom_lc**2, 1/sigma_odom_th_lc**2])
    edges_lc.append((i, i+1, z_n, Om_o))
    odom_lc.append(pose2_compose(odom_lc[-1], z_n))

odom_lc = np.array(odom_lc)
gt_lc = np.array(gt_poses_lc)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(gt_lc[:, 0], gt_lc[:, 1], 'forestgreen', lw=1.5, ls='--', alpha=0.5,
        marker='o', ms=3, label='Ground truth')
ax.plot(odom_lc[:, 0], odom_lc[:, 1], 'tomato', lw=2, marker='s', ms=3,
        label='Odometry (no LC)')

# Show the gap
gap = np.linalg.norm(odom_lc[-1, :2] - odom_lc[0, :2])
ax.annotate(f'gap = {gap:.2f} m', xy=odom_lc[-1, :2],
            xytext=(odom_lc[-1, 0]+1, odom_lc[-1, 1]+1),
            arrowprops=dict(arrowstyle='->', color='tomato', lw=2),
            fontsize=12, color='tomato', fontweight='bold')

ax.set_aspect('equal'); ax.legend(fontsize=11)
ax.set_title('Trajectory with accumulated drift (before loop closure)', fontsize=13)
plt.tight_layout()
plt.show()

## 26.3 Global Correction: How Optimization Distributes the Fix

Adding a loop closure edge and re-optimizing does not just fix the last
pose. The correction **propagates backward** through the entire trajectory.
The amount of correction at each pose depends on its position in the chain:
poses near the middle of the loop receive the largest shift.

This is because the optimizer finds the set of poses that best satisfies
ALL constraints simultaneously. The loop closure constraint forces the
end to match the beginning; the odometry constraints resist large changes
between consecutive poses. The result is a smooth distribution of the
correction.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
n_opt_iter_lc = 5
# ──────────────────────────────────────────────────────────────────────────────

# Case 1: No loop closure
edges_no_lc = edges_lc.copy()
init_no_lc = [odom_lc[k].copy() for k in range(n_poses_lc)]
opt_no_lc, costs_no_lc = pose_graph_optimize(init_no_lc, edges_no_lc, n_opt_iter_lc)
opt_no_lc_arr = np.array(opt_no_lc)

# Case 2: With loop closure
edges_with_lc = edges_lc.copy()
z_lc = pose2_between(gt_poses_lc[-1], gt_poses_lc[0])
z_lc += np.array([np.random.randn()*sigma_detect, np.random.randn()*sigma_detect,
                    np.random.randn()*sigma_detect_th])
z_lc[2] = wrap_angle(z_lc[2])
Om_lc = np.diag([1/sigma_detect**2, 1/sigma_detect**2, 1/sigma_detect_th**2])
edges_with_lc.append((n_poses_lc - 1, 0, z_lc, Om_lc))

init_with_lc = [odom_lc[k].copy() for k in range(n_poses_lc)]
opt_with_lc, costs_with_lc = pose_graph_optimize(init_with_lc, edges_with_lc, n_opt_iter_lc)
opt_with_lc_arr = np.array(opt_with_lc)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, opt_arr, title, color in [
    (axes[0], opt_no_lc_arr, 'Without loop closure', 'tomato'),
    (axes[1], opt_with_lc_arr, 'With loop closure', 'steelblue')]:
    ax.plot(gt_lc[:, 0], gt_lc[:, 1], 'forestgreen', lw=1.5, ls='--',
            alpha=0.5, label='Ground truth')
    ax.plot(opt_arr[:, 0], opt_arr[:, 1], color, lw=2, marker='o', ms=4,
            label='Optimized')
    ax.set_aspect('equal'); ax.legend(fontsize=10)
    ax.set_title(title, fontsize=13)

plt.suptitle('Loop closure: the trajectory snaps closed', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show per-pose correction (how much each pose moved)
corrections = np.linalg.norm(opt_with_lc_arr[:, :2] - odom_lc[:, :2], axis=1)
errors_no = [np.linalg.norm(opt_no_lc_arr[k, :2] - gt_lc[k, :2]) for k in range(n_poses_lc)]
errors_with = [np.linalg.norm(opt_with_lc_arr[k, :2] - gt_lc[k, :2]) for k in range(n_poses_lc)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(n_poses_lc), corrections, color='orange', alpha=0.7)
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Correction magnitude (m)', fontsize=12)
ax.set_title('Per-pose correction from loop closure', fontsize=13)

ax = axes[1]
ax.plot(errors_no, 'tomato', lw=2, marker='s', ms=4, label='Without LC')
ax.plot(errors_with, 'steelblue', lw=2, marker='o', ms=4, label='With LC')
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Error from ground truth (m)', fontsize=12)
ax.set_title('Per-pose error: with vs without loop closure', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print(f'Mean error without LC: {np.mean(errors_no):.4f} m')
print(f'Mean error with LC:    {np.mean(errors_with):.4f} m')
print(f'Largest correction at pose {np.argmax(corrections)}: {np.max(corrections):.4f} m')

**Key insight:** The correction is NOT applied only to the last pose.
It is distributed smoothly across the entire trajectory. Poses near the
middle of the loop (farthest from both the anchor and the loop closure
constraint) receive the largest correction. This is the "zipper" effect:
the entire path closes simultaneously.

---

## Capstone: Correct Loop Closure, Then False Loop Closure

First, we show the proper loop closure producing a beautiful correction.
Then, we add a **false loop closure** (connecting two poses that are NOT
actually at the same place) and show how it corrupts the entire map.
This motivates the need for robust verification and robust cost functions.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(55)
n_cap = 40
sigma_odom_cap = 0.1
sigma_odom_th_cap = 0.04
sigma_lc_cap = 0.02
sigma_lc_th_cap = 0.01
# ──────────────────────────────────────────────────────────────────────────────

# Ground truth: circular path
r_cap = 6.0
gt_angles_cap = np.linspace(0, 2*np.pi, n_cap, endpoint=False)
gt_cap = [np.array([r_cap*np.cos(a), r_cap*np.sin(a), a + np.pi/2]) for a in gt_angles_cap]
gt_cap_arr = np.array(gt_cap)

# Odometry edges
edges_base = []
odom_init = [gt_cap[0].copy()]
for i in range(n_cap - 1):
    z_t = pose2_between(gt_cap[i], gt_cap[i+1])
    noise = np.array([np.random.randn()*sigma_odom_cap, np.random.randn()*sigma_odom_cap,
                       np.random.randn()*sigma_odom_th_cap])
    z_n = z_t + noise; z_n[2] = wrap_angle(z_n[2])
    Om_o = np.diag([1/sigma_odom_cap**2, 1/sigma_odom_cap**2, 1/sigma_odom_th_cap**2])
    edges_base.append((i, i+1, z_n, Om_o))
    odom_init.append(pose2_compose(odom_init[-1], z_n))
odom_init = np.array(odom_init)

print(f'Ground truth: {n_cap} poses in a circle, radius = {r_cap} m')
print(f'Odometry drift at end: {np.linalg.norm(odom_init[-1,:2] - odom_init[0,:2]):.3f} m')

In [ ]:
# Scenario 1: No loop closure
init_1 = [odom_init[k].copy() for k in range(n_cap)]
opt_1, _ = pose_graph_optimize(init_1, edges_base, 5)
opt_1_arr = np.array(opt_1)

# Scenario 2: Correct loop closure
edges_correct = edges_base.copy()
z_lc_correct = pose2_between(gt_cap[-1], gt_cap[0])
z_lc_correct += np.array([np.random.randn()*sigma_lc_cap, np.random.randn()*sigma_lc_cap,
                            np.random.randn()*sigma_lc_th_cap])
z_lc_correct[2] = wrap_angle(z_lc_correct[2])
Om_lc_cap = np.diag([1/sigma_lc_cap**2, 1/sigma_lc_cap**2, 1/sigma_lc_th_cap**2])
edges_correct.append((n_cap - 1, 0, z_lc_correct, Om_lc_cap))

init_2 = [odom_init[k].copy() for k in range(n_cap)]
opt_2, _ = pose_graph_optimize(init_2, edges_correct, 5)
opt_2_arr = np.array(opt_2)

# Scenario 3: FALSE loop closure (connecting pose 20 to pose 35, which are NOT the same place)
edges_false = edges_correct.copy()  # includes the correct LC too
false_i = 10
false_j = 30
# The false measurement claims these two poses are at the same place (zero relative transform)
z_false = np.array([0.0, 0.0, 0.0])  # completely wrong!
Om_false = np.diag([1/sigma_lc_cap**2, 1/sigma_lc_cap**2, 1/sigma_lc_th_cap**2])
edges_false.append((false_i, false_j, z_false, Om_false))

init_3 = [odom_init[k].copy() for k in range(n_cap)]
opt_3, _ = pose_graph_optimize(init_3, edges_false, 5)
opt_3_arr = np.array(opt_3)

print(f'Scenario 1: no loop closure')
print(f'Scenario 2: correct loop closure (pose {n_cap-1} -> pose 0)')
print(f'Scenario 3: correct LC + FALSE LC (pose {false_i} -> pose {false_j})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, opt_arr, title, color in [
    (axes[0], opt_1_arr, 'No loop closure\n(drifted)', 'tomato'),
    (axes[1], opt_2_arr, 'Correct loop closure\n(fixed!)', 'steelblue'),
    (axes[2], opt_3_arr, 'Correct + FALSE LC\n(corrupted!)', 'orange')]:
    
    ax.plot(gt_cap_arr[:, 0], gt_cap_arr[:, 1], 'forestgreen', lw=1.5, ls='--',
            alpha=0.4, label='Ground truth')
    ax.plot(opt_arr[:, 0], opt_arr[:, 1], color, lw=2, marker='o', ms=3,
            label='Optimized')
    ax.set_aspect('equal'); ax.legend(fontsize=9, loc='lower left')
    ax.set_title(title, fontsize=13)
    ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)

# Mark false LC
ax = axes[2]
ax.plot([opt_3_arr[false_i, 0], opt_3_arr[false_j, 0]],
        [opt_3_arr[false_i, 1], opt_3_arr[false_j, 1]],
        'r-', lw=3, zorder=10)
ax.annotate('FALSE', xy=((opt_3_arr[false_i, 0]+opt_3_arr[false_j, 0])/2,
                          (opt_3_arr[false_i, 1]+opt_3_arr[false_j, 1])/2),
            fontsize=14, color='red', fontweight='bold', ha='center')

plt.suptitle('Loop closure: correct vs false', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Error comparison
err_1 = [np.linalg.norm(opt_1_arr[k,:2] - gt_cap_arr[k,:2]) for k in range(n_cap)]
err_2 = [np.linalg.norm(opt_2_arr[k,:2] - gt_cap_arr[k,:2]) for k in range(n_cap)]
err_3 = [np.linalg.norm(opt_3_arr[k,:2] - gt_cap_arr[k,:2]) for k in range(n_cap)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(err_1, 'tomato', lw=2, marker='s', ms=3, label='No LC')
ax.plot(err_2, 'steelblue', lw=2, marker='o', ms=3, label='Correct LC')
ax.plot(err_3, 'orange', lw=2, marker='d', ms=3, label='Correct + False LC')
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Per-pose error: three scenarios', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Summary:')
print(f'  No LC:              mean error = {np.mean(err_1):.4f} m')
print(f'  Correct LC:         mean error = {np.mean(err_2):.4f} m')
print(f'  Correct + False LC: mean error = {np.mean(err_3):.4f} m')
print(f'\nA single false loop closure can corrupt the ENTIRE map.')
print(f'This is why robust verification and robust cost functions are essential.')

In [ ]:
# Mitigation: use Huber cost to downweight the false loop closure
def pose_graph_optimize_robust(poses_init, edges, n_iter=5, delta=1.0):
    """Gauss-Newton with Huber weighting for robustness."""
    n = len(poses_init)
    poses = [p.copy() for p in poses_init]
    costs = []
    for it in range(n_iter):
        dim = 3 * n; H = np.zeros((dim, dim)); b = np.zeros(dim)
        cost = 0.0
        for (i, j, z_ij, Om) in edges:
            e = compute_edge_residual(poses[i], poses[j], z_ij)
            A, B = compute_edge_jacobians(poses[i], poses[j])
            
            # Huber weighting
            e_norm = np.sqrt(e @ Om @ e)
            if e_norm <= delta:
                w = 1.0
            else:
                w = delta / e_norm
            
            cost += w * e @ Om @ e
            Om_w = w * Om
            
            ri, rj = 3*i, 3*j
            H[ri:ri+3, ri:ri+3] += A.T @ Om_w @ A
            H[ri:ri+3, rj:rj+3] += A.T @ Om_w @ B
            H[rj:rj+3, ri:ri+3] += B.T @ Om_w @ A
            H[rj:rj+3, rj:rj+3] += B.T @ Om_w @ B
            b[ri:ri+3] += A.T @ Om_w @ e
            b[rj:rj+3] += B.T @ Om_w @ e
        costs.append(cost)
        H[:3,:] = 0; H[:,:3] = 0; H[:3,:3] = np.eye(3)*1e6; b[:3] = 0
        dx = np.linalg.solve(H, -b)
        for k in range(n):
            poses[k] += dx[3*k:3*k+3]; poses[k][2] = wrap_angle(poses[k][2])
    return poses, costs

# Optimize with Huber cost
init_robust = [odom_init[k].copy() for k in range(n_cap)]
opt_robust, costs_robust = pose_graph_optimize_robust(init_robust, edges_false, 10, delta=2.0)
opt_robust_arr = np.array(opt_robust)

err_robust = [np.linalg.norm(opt_robust_arr[k,:2] - gt_cap_arr[k,:2]) for k in range(n_cap)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(gt_cap_arr[:, 0], gt_cap_arr[:, 1], 'forestgreen', lw=1.5, ls='--', alpha=0.4, label='Truth')
ax.plot(opt_3_arr[:, 0], opt_3_arr[:, 1], 'orange', lw=1.5, alpha=0.4, label='Standard (corrupted)')
ax.plot(opt_robust_arr[:, 0], opt_robust_arr[:, 1], 'steelblue', lw=2, marker='o', ms=3,
        label='Robust (Huber)')
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Robust cost resists false loop closure', fontsize=13)
ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)

ax = axes[1]
ax.plot(err_3, 'orange', lw=2, marker='d', ms=3, label='Standard (false LC corrupts)')
ax.plot(err_robust, 'steelblue', lw=2, marker='o', ms=3, label='Robust (Huber, false LC rejected)')
ax.plot(err_2, 'forestgreen', lw=1, ls='--', alpha=0.5, label='Correct LC only (reference)')
ax.set_xlabel('Pose index', fontsize=12)
ax.set_ylabel('Error (m)', fontsize=12)
ax.set_title('Robust cost limits damage from false LC', fontsize=13)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Standard optimizer with false LC: mean error = {np.mean(err_3):.4f} m')
print(f'Robust optimizer with false LC:   mean error = {np.mean(err_robust):.4f} m')

**Capstone observations:**
- **No loop closure:** Drift accumulates, the trajectory does not close.
- **Correct loop closure:** The optimizer distributes the correction smoothly across all poses. The entire trajectory snaps to match the ground truth.
- **False loop closure:** A single incorrect constraint warps the entire map, pulling two distant parts of the trajectory together and distorting everything in between.
- **Robust cost functions** (Huber) can mitigate the damage by downweighting the outlier constraint. The optimizer effectively ignores the false loop closure while still benefiting from the correct one.
- This is why loop closure verification is one of the most critical components in any SLAM system. A single false positive can destroy an otherwise perfect map.

---

## Exercises

### Exercise 26.1: Multiple loop closures

Add loop closure edges at 3 different points around the circle
(e.g., pose 10 to pose 0, pose 20 to pose 10, pose 30 to pose 0).
How does having multiple loop closures improve accuracy compared to
having just one?

In [ ]:
# Your code here

### Exercise 26.2: Detection threshold

Implement a simple loop closure detector: for each new pose, check if any
previous pose is within distance $d$ (using the current noisy estimate).
Sweep $d$ from 0.5 to 5.0 m and count true positives vs false positives.
Plot a precision-recall curve.

In [ ]:
# Your code here

### Exercise 26.3: Robust cost comparison (challenge)

Run the false loop closure scenario with three different cost functions:
squared (standard), Huber, and Cauchy. Compare the resulting trajectories
and errors. Which cost function is most resistant to the false loop closure?

In [ ]:
# Your code here
# Hint: for Cauchy cost, weight = 1 / (1 + (e_norm/delta)^2)